# SkyID — Training Notebook
**EfficientNet-B3 fine-tuned on the 200-class combined aircraft dataset**

### Before running:
1. In the top menu go to **Runtime → Change runtime type** and select **T4 GPU**
2. Upload `combined.zip` to your Google Drive inside a folder called `SkyID`
3. Run all cells top to bottom

Checkpoints are saved to Drive after every epoch — if the session disconnects, just re-run from **Cell 4** onwards and set `START_EPOCH` to where you left off.

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/SkyID'
print(f'Drive mounted. Working folder: {DRIVE_DIR}')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q timm safetensors
print('Dependencies ready.')

In [ ]:
# ── Cell 3: Extract dataset ───────────────────────────────────────────────────
import os, zipfile

ZIP_PATH    = f'{DRIVE_DIR}/combined.zip'
DATASET_DIR = '/content/combined'

if os.path.exists(DATASET_DIR):
    print('Dataset already extracted — skipping.')
else:
    print('Extracting dataset... (this takes 2-5 minutes)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    print('Done!')

classes = sorted(os.listdir(DATASET_DIR))
total   = sum(len(os.listdir(os.path.join(DATASET_DIR, c))) for c in classes)
print(f'{len(classes)} classes | {total:,} images')

In [ ]:
# ── Cell 4: Configuration ─────────────────────────────────────────────────────
# Edit these if you want to change training settings

BATCH_SIZE          = 32
EPOCHS              = 60
WARMUP_EPOCHS       = 5
EARLY_STOP_PATIENCE = 12
START_EPOCH         = 0   # Change to resume (e.g. 10 to resume from epoch 10)

CHECKPOINT_DIR = f'{DRIVE_DIR}/checkpoints'
BEST_MODEL     = f'{DRIVE_DIR}/skyid_combined.pth'
CLASSES_FILE   = f'{DRIVE_DIR}/combined_classes.txt'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {CHECKPOINT_DIR}')
print(f'Best model will be saved to:  {BEST_MODEL}')

In [ ]:
# ── Cell 5: Build data loaders ────────────────────────────────────────────────
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import ImageFolder

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print(f'Training on: {DEVICE}  |  AMP: {USE_AMP}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

train_transforms = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.RandomCrop(300),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

val_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_full = ImageFolder(DATASET_DIR, transform=train_transforms)
val_full   = ImageFolder(DATASET_DIR, transform=val_transforms)

num_classes = len(train_full.classes)
print(f'\n{num_classes} classes | {len(train_full):,} total images')

with open(CLASSES_FILE, 'w') as f:
    for cls in train_full.classes:
        f.write(cls + '\n')
print('Class names saved to Drive')

# 85/15 split with fixed seed
val_size   = int(0.15 * len(train_full))
train_size = len(train_full) - val_size
g1 = torch.Generator().manual_seed(42)
g2 = torch.Generator().manual_seed(42)
train_idx, _ = torch.utils.data.random_split(range(len(train_full)), [train_size, val_size], generator=g1)
_, val_idx   = torch.utils.data.random_split(range(len(val_full)),   [train_size, val_size], generator=g2)

train_loader = DataLoader(Subset(train_full, train_idx.indices), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(Subset(val_full,   val_idx.indices),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Cell 6: Build model ───────────────────────────────────────────────────────
import timm
import torch.nn as nn

# drop_path_rate adds stochastic depth — strong anti-overfitting for EfficientNet
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes,
                          drop_path_rate=0.3)

in_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4),
    nn.Linear(in_features, num_classes)
)
model = model.to(DEVICE)

# Resume from checkpoint if START_EPOCH > 0
resume_path = f'{CHECKPOINT_DIR}/epoch_{START_EPOCH:02d}.pth'
if START_EPOCH > 0 and os.path.exists(resume_path):
    model.load_state_dict(torch.load(resume_path, map_location=DEVICE))
    print(f'Resumed from checkpoint: epoch {START_EPOCH}')
elif START_EPOCH > 0:
    print(f'WARNING: No checkpoint found at {resume_path}, starting from scratch')
else:
    print('Starting fresh with ImageNet pretrained weights')

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
# ── Cell 7: Train ─────────────────────────────────────────────────────────────
import math, time

backbone_params   = [p for n, p in model.named_parameters() if 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'classifier' in n]
optimizer = torch.optim.AdamW([
    {'params': backbone_params,   'lr': 1e-5},
    {'params': classifier_params, 'lr': 1e-3},
], weight_decay=1e-3)

# Linear warmup then cosine decay
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler    = torch.amp.GradScaler('cuda') if USE_AMP else None

# Fast-forward scheduler if resuming
for _ in range(START_EPOCH):
    scheduler.step()

# Load best acc from log if resuming
best_acc       = 0.0
patience_count = 0
log_file = f'{DRIVE_DIR}/training_log.txt'
if os.path.exists(log_file) and START_EPOCH > 0:
    with open(log_file) as f:
        lines = [l for l in f.readlines() if 'best=' in l]
    if lines:
        try:
            best_acc = float(lines[-1].strip().split('best=')[-1])
            print(f'Resuming with best_acc = {best_acc:.4f}')
        except:
            pass

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    outputs = model(imgs)
            else:
                outputs = model(imgs)
            correct += (outputs.argmax(1) == labels).sum().item()
            total   += labels.size(0)
    return correct / total

print(f'Starting training from epoch {START_EPOCH + 1}\n')

for epoch in range(START_EPOCH, EPOCHS):
    t0 = time.time()
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for i, (imgs, labels) in enumerate(train_loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        if USE_AMP:
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
                loss    = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)

        if (i + 1) % 100 == 0:
            print(f'  Epoch {epoch+1} | Batch {i+1}/{len(train_loader)} | '
                  f'Loss: {running_loss/(i+1):.3f} | Acc: {correct/total:.3f}', end='\r')

    train_acc = correct / total
    val_acc   = evaluate(model, val_loader)
    scheduler.step()
    elapsed = time.time() - t0

    log_line = (f'Epoch {epoch+1:02d}/{EPOCHS} | '
                f'Loss: {running_loss/len(train_loader):.3f} | '
                f'Train: {train_acc:.4f} | '
                f'Val: {val_acc:.4f} | '
                f'Time: {elapsed/60:.1f}m')
    print(f'\n{log_line}')

    # Save per-epoch checkpoint to Drive (for resuming)
    ckpt_path = f'{CHECKPOINT_DIR}/epoch_{epoch+1:02d}.pth'
    torch.save(model.state_dict(), ckpt_path)

    if val_acc > best_acc:
        best_acc       = val_acc
        patience_count = 0
        torch.save(model.state_dict(), BEST_MODEL)
        print(f'  ★ New best! val_acc={best_acc:.4f} → saved to Drive')
    else:
        patience_count += 1
        print(f'  No improvement ({patience_count}/{EARLY_STOP_PATIENCE})')
        if patience_count >= EARLY_STOP_PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}')
            break

    with open(log_file, 'a') as f:
        f.write(log_line + f' | best={best_acc:.4f}\n')

print(f'\nTraining complete. Best val accuracy: {best_acc:.4f}')
if best_acc < 0.75:
    print('  WARNING: Did not reach 75% target.')
else:
    print('  Target of 75% reached!')

In [ ]:
# ── Cell 8: Download model to your computer ───────────────────────────────────
# Run this cell to download skyid_combined.pth directly to your browser downloads
from google.colab import files
files.download(BEST_MODEL)
files.download(CLASSES_FILE)